In [1]:
import gurobipy as gp

RUTA_LP = "chile_censal_reducido/modelo_chile_censal_eps_0.37000_A_sl_v2.lp"

modelo_con_limite = gp.read(RUTA_LP)

print("Variables:", modelo_con_limite.NumVars)
print("Restricciones:", modelo_con_limite.NumConstrs)

Set parameter Username
Set parameter LicenseID to value 2819129
Academic license - for non-commercial use only - expires 2027-05-06
Read LP format model from file chile_censal_reducido/modelo_chile_censal_eps_0.37000_A_sl_v2.lp
Reading time = 14.34 seconds
: 1911217 rows, 319466 columns, 4756616 nonzeros
Variables: 319466
Restricciones: 1911217


In [2]:
from funciones import build_matrices_from_gurobi

In [3]:
A_leq, b_leq, C, d, meta = build_matrices_from_gurobi(
    modelo_con_limite,
    include_bounds=True
)

print("A_leq:", A_leq.shape)
print("b_leq:", b_leq.shape)
print("C:", C.shape)
print("d:", d.shape)

A_leq: (2545108, 319466)
b_leq: (2545108,)
C: (5041, 319466)
d: (5041,)


In [4]:
var_names = meta["var_names"]

idx_X = [
    k for k, nombre in enumerate(var_names)
    if nombre.startswith("asignaciones_ij")
]

idx_y = [
    k for k, nombre in enumerate(var_names)
    if nombre.startswith("centros_j")
]

print("Variables X:", len(idx_X))
print("Variables y:", len(idx_y))
print("Total:", len(var_names))

print("\nEjemplo X:")
print([var_names[k] for k in idx_X[:5]])

print("\nEjemplo y:")
print([var_names[k] for k in idx_y[:5]])

Variables X: 316946
Variables y: 2520
Total: 319466

Ejemplo X:
['asignaciones_ij[dist_210407,dist_210405]', 'asignaciones_ij[dist_210402,dist_210405]', 'asignaciones_ij[dist_210401,dist_210405]', 'asignaciones_ij[dist_210404,dist_210405]', 'asignaciones_ij[dist_210403,dist_210405]']

Ejemplo y:
['centros_j[dist_210405]', 'centros_j[dist_210407]', 'centros_j[dist_210408]', 'centros_j[dist_210402]', 'centros_j[dist_210401]']


In [5]:
import json
import numpy as np

RUTA_JSON = r"chile_censal_reducido/valores_chile_censal_eps_0.37000_A_sl_v2.json"

with open(RUTA_JSON, "r", encoding="utf-8") as f:
    valores = json.load(f)

# Verificar que los nombres coincidan
faltantes = [nombre for nombre in var_names if nombre not in valores]

print("Variables sin valor en JSON:", len(faltantes))
print("Ejemplos:", faltantes[:5])

Variables sin valor en JSON: 0
Ejemplos: []


In [6]:
x1_full = np.array(
    [float(valores[nombre]) for nombre in var_names],
    dtype=float
)

X1 = x1_full[idx_X]
y1 = x1_full[idx_y]

print("Tamaño X1:", len(X1))
print("Tamaño y1:", len(y1))
print("Suma y1:", y1.sum())
print("X1 min/max:", X1.min(), X1.max())
print("y1 min/max:", y1.min(), y1.max())

Tamaño X1: 316946
Tamaño y1: 2520
Suma y1: 27.99999999999646
X1 min/max: -1.5046027492982407e-11 0.9874057135378732
y1 min/max: -4.942710059415768e-14 0.9874057135378732


In [7]:
# Separar columnas X e y
A_X = A_leq[:, idx_X]
A_y = A_leq[:, idx_y]

C_X = C[:, idx_X]
C_y = C[:, idx_y]

# Sustituir y = y1
b1_X = b_leq - A_y @ y1
d_X  = d     - C_y @ y1

print("A_X:", A_X.shape)
print("b1_X:", b1_X.shape)
print("C_X:", C_X.shape)
print("d_X:", d_X.shape)

A_X: (2545108, 316946)
b1_X: (2545108,)
C_X: (5041, 316946)
d_X: (5041,)


In [8]:
viol_ineq = np.max(A_X @ X1 - b1_X)
viol_eq   = np.max(np.abs(C_X @ X1 - d_X))

print("Máxima violación desigualdades:", viol_ineq)
print("Máxima violación igualdades:", viol_eq)

Máxima violación desigualdades: 3.766918739167471e-08
Máxima violación igualdades: 4.338480685817103e-10


In [9]:
epsilon0 = 0.37
epsilon1 = 0.40

In [10]:
nombres_constr = meta["constr_names"]

pop_up = [
    (k, nombre)
    for k, nombre in enumerate(nombres_constr)
    if nombre.startswith("pop_up[")
]

pop_lo = [
    (k, nombre)
    for k, nombre in enumerate(nombres_constr)
    if nombre.startswith("pop_lo[")
]

print("pop_up:", len(pop_up))
print("pop_lo:", len(pop_lo))

print(pop_up[:3])
print(pop_lo[:3])

pop_up: 2520
pop_lo: 2520
[(0, 'pop_up[dist_210405]'), (3, 'pop_up[dist_210407]'), (6, 'pop_up[dist_210408]')]
[(1, 'pop_lo[dist_210405]'), (4, 'pop_lo[dist_210407]'), (7, 'pop_lo[dist_210408]')]


In [11]:
b2_X = b1_X.copy()

# Balance superior
for k_global, nombre in pop_up:
    fila = meta["row_map_leq"][k_global]
    b2_X[fila] = b1_X[fila] * (1 + epsilon1) / (1 + epsilon0)

# Balance inferior
for k_global, nombre in pop_lo:
    fila = meta["row_map_leq"][k_global]
    b2_X[fila] = b1_X[fila] * (1 - epsilon1) / (1 - epsilon0)

In [12]:
cambio = b2_X - b1_X

print("Filas modificadas:", np.sum(np.abs(cambio) > 1e-10))
print("Cambio mínimo:", cambio.min())
print("Cambio máximo:", cambio.max())

Filas modificadas: 5040
Cambio mínimo: -9.371976164044148e-10
Cambio máximo: 18722.40673694655


In [13]:
# --------------------------------------------------
# Eliminar filas que quedaron sin variables X
# --------------------------------------------------

# Desigualdades
nnz_A = np.diff(A_X.indptr)
keep_A = nnz_A > 0

A_X2 = A_X[keep_A, :]
b2_X2 = b2_X[keep_A]

# Igualdades
nnz_C = np.diff(C_X.indptr)
keep_C = nnz_C > 0

C_X2 = C_X[keep_C, :]
d_X2 = d_X[keep_C]

print("A antes/después:", A_X.shape, A_X2.shape)
print("C antes/después:", C_X.shape, C_X2.shape)
print("Filas constantes A eliminadas:", np.sum(~keep_A))
print("Filas constantes C eliminadas:", np.sum(~keep_C))

A antes/después: (2545108, 316946) (2540068, 316946)
C antes/después: (5041, 316946) (5040, 316946)
Filas constantes A eliminadas: 5040
Filas constantes C eliminadas: 1


In [14]:
print(
    "Violación A:",
    np.max(A_X2 @ X1 - b2_X2)
)

print(
    "Violación C:",
    np.max(np.abs(C_X2 @ X1 - d_X2))
)

Violación A: 2.0502677557899873e-08
Violación C: 4.338480685817103e-10


In [15]:
from gurobipy import Model, GRB
import numpy as np

N = len(y1)

X1_clean = np.clip(X1, 0.0, 1.0)

m = Model("cota_L1")

m.Params.OutputFlag = 1
m.Params.Method = 2
m.Params.Crossover = 0

x = m.addMVar(
    len(X1_clean),
    lb=0.0,
    ub=1.0,
    name="x"
)

m.addConstr(A_X2 @ x <= b2_X2)
m.addConstr(C_X2 @ x == d_X2)

coef = 1.0 - 2.0 * X1_clean
constante = float(X1_clean.sum())

m.setObjective(
    constante + coef @ x,
    GRB.MAXIMIZE
)

m.optimize()

Set parameter OutputFlag to value 1
Set parameter Method to value 2
Set parameter Crossover to value 0
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1355U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
Method  2
Crossover  0

Optimize a model with 2545108 rows, 316946 columns and 5063482 nonzeros
Model fingerprint: 0xf99d3c03
Coefficient statistics:
  Matrix range     [1e+00, 1e+05]
  Objective range  [2e-03, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e-15, 9e+05]
Presolve removed 58975 rows and 59015 columns
Presolve time: 0.37s

Barrier solved model in 0 iterations and 0.37 seconds (0.27 work units)
Model is infeasible


In [16]:
X1_clean = np.clip(X1, 0.0, 1.0)

viol_A_clean = np.max(A_X2 @ X1_clean - b2_X2)
viol_C_clean = np.max(np.abs(C_X2 @ X1_clean - d_X2))

print("Violación A con X1_clean:", viol_A_clean)
print("Violación C con X1_clean:", viol_C_clean)

print("RHS mínimo A:", b2_X2.min())
print("RHS mínimo abs C:", np.min(np.abs(d_X2)))

Violación A con X1_clean: 4.535327601494531e-08
Violación C con X1_clean: 5.534614988533804e-10
RHS mínimo A: -374448.1347389312
RHS mínimo abs C: 6.342007815185761e-15


In [17]:
from gurobipy import Model, GRB

m_test = Model("test_factibilidad")

m_test.Params.OutputFlag = 1
m_test.Params.Presolve = 0
m_test.Params.Method = 2
m_test.Params.Crossover = 0
m_test.Params.NumericFocus = 3

x_test = m_test.addMVar(
    len(X1_clean),
    lb=0.0,
    ub=1.0,
    name="x"
)

m_test.addConstr(A_X2 @ x_test <= b2_X2)
m_test.addConstr(C_X2 @ x_test == d_X2)

# Solo factibilidad
m_test.setObjective(0.0, GRB.MINIMIZE)

m_test.optimize()

print("Status:", m_test.Status)

Set parameter OutputFlag to value 1
Set parameter Presolve to value 0
Set parameter Method to value 2
Set parameter Crossover to value 0
Set parameter NumericFocus to value 3
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1355U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
Method  2
Crossover  0
NumericFocus  3
Presolve  0

Optimize a model with 2545108 rows, 316946 columns and 5063482 nonzeros
Model fingerprint: 0xb56bf77e
Coefficient statistics:
  Matrix range     [1e+00, 1e+05]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e-15, 9e+05]
Elapsed ordering time = 5s
Elapsed ordering time = 10s
Ordering time: 17.08s

Barrier statistics:
 AA' NZ     : 5.993e+07
 Factor NZ  : 2.256e+08 (roughly 3.0 GB of memory)
 Factor Ops : 5.690e+10 (roughly 4 seconds per iteration)
 Threads    

In [18]:
# Cota lineal
coef = 1.0 - 2.0 * X1_clean
constante = float(X1_clean.sum())

m_test.setObjective(
    constante + coef @ x_test,
    GRB.MAXIMIZE
)

m_test.optimize()

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1355U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
Method  2
Crossover  0
GomoryPasses  0
NumericFocus  3
Presolve  0

Optimize a model with 2545108 rows, 316946 columns and 5063482 nonzeros
Model fingerprint: 0xf99d3c03
Coefficient statistics:
  Matrix range     [1e+00, 1e+05]
  Objective range  [2e-03, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e-15, 9e+05]
Elapsed ordering time = 5s
Elapsed ordering time = 10s
Ordering time: 18.67s

Barrier statistics:
 AA' NZ     : 5.993e+07
 Factor NZ  : 2.256e+08 (roughly 3.0 GB of memory)
 Factor Ops : 5.691e+10 (roughly 3 seconds per iteration)
 Threads    : 10

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl     Time
   0   1.12190872e+08  2.52

In [19]:
if m_test.Status == GRB.OPTIMAL:

    x_hat = np.array(x_test.X)

    U = float(m_test.ObjVal)
    L1_real = float(np.sum(np.abs(x_hat - X1_clean)))

    N = len(y1)

    print("U =", U)
    print("U / (2N) =", U / (2 * N))

    print("L1 real del x encontrado =", L1_real)
    print("L1 real / (2N) =", L1_real / (2 * N))

    print("Holgura =", U - L1_real)
    print("Holgura normalizada =", (U - L1_real) / (2 * N))

U = 4206.8901777314795
U / (2N) = 0.8347004320895792
L1 real del x encontrado = 1226.0833625987223
L1 real / (2N) = 0.24327050845212744
Holgura = 2980.806815132757
Holgura normalizada = 0.5914299236374518


In [20]:
U / (2 * N)

0.8347004320895792

In [21]:
L1_real / (2 * N)

0.24327050845212744

In [22]:
tol = 1e-6

n_frac = np.sum(
    (X1_clean > tol) &
    (X1_clean < 1 - tol)
)

print("Componentes X:", len(X1_clean))
print("Fraccionarias:", n_frac)
print("Proporción:", n_frac / len(X1_clean))

Componentes X: 316946
Fraccionarias: 250397
Proporción: 0.7900304783780202


In [23]:
U_base = np.sum(2 * X1_clean * (1 - X1_clean))

print("Surrogate en X*= ", U_base)
print("Surrogate normalizado en X* =", U_base / (2 * N))

Surrogate en X*=  4037.9035534229843
Surrogate normalizado en X* = 0.8011713399648779


In [24]:
k = 0

# máximo de x_k
m_test.setObjective(x_test[k], GRB.MAXIMIZE)
m_test.optimize()

U_k = x_test[k].X

# mínimo de x_k
m_test.setObjective(x_test[k], GRB.MINIMIZE)
m_test.optimize()

L_k = x_test[k].X

Delta_k = max(
    U_k - X1_clean[k],
    X1_clean[k] - L_k
)

print("x*_k =", X1_clean[k])
print("L_k  =", L_k)
print("U_k  =", U_k)
print("Delta_k =", Delta_k)

Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1355U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
Method  2
Crossover  0
GomoryPasses  0
NumericFocus  3
Presolve  0

Optimize a model with 2545108 rows, 316946 columns and 5063482 nonzeros
Model fingerprint: 0x2a93df3b
Coefficient statistics:
  Matrix range     [1e+00, 1e+05]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e-15, 9e+05]
Elapsed ordering time = 5s
Elapsed ordering time = 10s
Elapsed ordering time = 15s
Elapsed ordering time = 20s
Elapsed ordering time = 25s
Ordering time: 28.68s

Barrier statistics:
 AA' NZ     : 5.993e+07
 Factor NZ  : 2.256e+08 (roughly 3.0 GB of memory)
 Factor Ops : 5.690e+10 (roughly 4 seconds per iteration)
 Threads    : 10

                  Objective                Residual
Iter       Primal

In [25]:
objetivos = [0.01, 0.25, 0.50, 0.75, 0.95]

indices_prueba = [
    np.argmin(np.abs(X1_clean - valor))
    for valor in objetivos
]

for k in indices_prueba:
    print(k, X1_clean[k])

239429 0.009999597756116984
168015 0.24991476164577953
161403 0.4988030609478824
107274 0.7469702882295306
161454 0.967889329234692


In [ ]:
import time
import numpy as np
from gurobipy import GRB

# Cambiamos a primal simplex para aprovechar reoptimizaciones
m_test.Params.Method = 0
m_test.Params.Presolve = 0
m_test.Params.OutputFlag = 0

# Los 5 valores representativos que encontramos
indices_test = [
    239429,   # x* ~ 0.01
    168015,   # x* ~ 0.25
    161403,   # x* ~ 0.50
    107274,   # x* ~ 0.75
    161454    # x* ~ 0.97
]

resultados = []

for k in indices_test:

    print(f"\n--- k={k}, x*={X1_clean[k]:.6f} ---")

    # MAX x_k
    t0 = time.time()

    m_test.setObjective(x_test[k], GRB.MAXIMIZE)
    m_test.optimize()

    tiempo_max = time.time() - t0

    if m_test.Status != GRB.OPTIMAL:
        print("MAX no óptimo. Status:", m_test.Status)
        continue

    U_k = float(x_test.X[k])

    # MIN x_k
    t0 = time.time()

    m_test.setObjective(x_test[k], GRB.MINIMIZE)
    m_test.optimize()

    tiempo_min = time.time() - t0

    if m_test.Status != GRB.OPTIMAL:
        print("MIN no óptimo. Status:", m_test.Status)
        continue

    L_k = float(x_test.X[k])

    Delta_k = max(
        U_k - X1_clean[k],
        X1_clean[k] - L_k
    )

    resultados.append({
        "k": k,
        "x_star": X1_clean[k],
        "L": L_k,
        "U": U_k,
        "Delta": Delta_k,
        "t_max": tiempo_max,
        "t_min": tiempo_min
    })

    print("L       =", L_k)
    print("U       =", U_k)
    print("Delta   =", Delta_k)
    print("t MAX   =", tiempo_max, "s")
    print("t MIN   =", tiempo_min, "s")

Set parameter Method to value 0
Set parameter Presolve to value 0

--- k=239429, x*=0.010000 ---
